# AfyaFlow Pwani: Gemma 4 Stock-Out Early Warning Demo

AfyaFlow Pwani targets Track 3: Smart Health - AI-Driven Health Center & Supply Chain Management. The prototype focuses on one high-risk operational gap: medicine and diagnostic-test stock-outs that remain hidden in paper cards, phone calls, or fragmented reports until patients are already affected.

This notebook demonstrates the reproducible report-to-alert workflow with synthetic data only. In Kaggle GPU mode, the `GemmaClient` runtime boundary can be connected to a real Gemma 4 model while preserving the same validation, deterministic risk, approved tool, and audit layers.

## Why Gemma 4 is central

Gemma 4 is intended to handle the unstructured part of the workflow: multilingual facility reports, stock-card images, short voice reports, strict JSON extraction, and constrained tool calling. AfyaFlow deliberately keeps arithmetic and redistribution logic deterministic so the model cannot hallucinate operational decisions.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if (ROOT / 'src').exists():
    sys.path.insert(0, str(ROOT / 'src'))
elif (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
    sys.path.insert(0, str(ROOT / 'src'))

print('Notebook root:', ROOT)

In [ ]:
from dataclasses import asdict

from afyaflow.data_loader import load_facility_inventory, load_stock_report_examples, stock_report_from_expected
from afyaflow.extraction import build_extraction_prompt, extract_with_rules
from afyaflow.risk_engine import calculate_stock_risk_with_transfers
from afyaflow.tool_registry import ToolCall, execute_tool_call
from afyaflow.workflow import run_report_workflow

inventory_path = ROOT / 'data' / 'synthetic' / 'facility_inventory.json'
reports_path = ROOT / 'data' / 'synthetic' / 'stock_reports.json'
inventory = load_facility_inventory(inventory_path)
examples = load_stock_report_examples(reports_path)
len(inventory), len(examples)

## Synthetic golden scenario

The golden demo starts with a mixed Swahili/English facility report about ORS sachets. The notebook uses deterministic fallback extraction locally so the workflow is testable without model weights. On Kaggle GPU, the same `GemmaClient` boundary should be connected to a real Gemma 4 runtime.

In [ ]:
example = examples[0]
raw_input = example['raw_input']
print(raw_input)
print()
print(build_extraction_prompt(raw_input, 'text'))

In [ ]:
report = extract_with_rules(raw_input)
asdict(report)

In [ ]:
risk = calculate_stock_risk_with_transfers(report, inventory)
asdict(risk)

## Approved tool calling

Gemma should only be allowed to request approved operational tools. The application validates each call before deterministic code executes it.

In [ ]:
handoff = execute_tool_call(
    ToolCall(name='draft_handoff_message', arguments={'language': 'sw'}),
    report,
    inventory,
)
handoff

In [ ]:
result = run_report_workflow(
    raw_input,
    inventory_path=inventory_path,
    handoff_language='sw',
)
{
    'facility': result['report'].facility,
    'item': result['report'].item,
    'risk_level': result['risk'].level,
    'days_of_stock': result['risk'].days_of_stock,
    'handoff': result['handoff'],
}

## Mini evaluation

This smoke evaluation verifies that the public synthetic examples produce the expected deterministic risk bands after extraction. A larger frozen benchmark will be added in the evaluation milestone.

In [ ]:
rows = []
for item in examples:
    expected_report = stock_report_from_expected(item)
    expected_risk = calculate_stock_risk_with_transfers(expected_report, inventory)
    rows.append({
        'id': item['id'],
        'facility': expected_report.facility,
        'item': expected_report.item,
        'expected': item['expected']['risk_level'],
        'actual': expected_risk.level,
        'pass': item['expected']['risk_level'] == expected_risk.level,
    })
rows

## Kaggle GPU Gemma runtime hook

In the competition notebook, connect a real Gemma 4 runtime behind `ModelRuntime.generate_json(prompt)`. Keep the downstream schema validation, human confirmation, deterministic risk engine, approved tool registry, and audit trail unchanged. This separation is the safety story: Gemma understands messy reports; AfyaFlow controls the operational action.